# Step 4: Train JMM Oracle

Prediction head pretraining + joint fine-tuning + Optuna HPO.

In [1]:
from molrl.dataloader import create_dataloader
from molrl.nnx_modules import SmilesEncoder, PredictionHead
from molrl.models import EncoderPredictor
from molrl.training import oracle_train_step, oracle_val_step
from flax import nnx
from jax import numpy as jnp
import numpy as np
import optax
import orbax.checkpoint as ocp
from pathlib import Path
import random
import pandas as pd
import optuna

/Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
base_config = {
    "dataset": "chembl224_ki",
    "batch_size": 256,
    "n_epochs": 2000,
    "early_stop_patience_evals": 5,
    "model_save_path": "checkpoints/pretrained_oracle",
    "vocab_size": 36,
    "max_seq_len": 102,
    "latent_dim": 128,
    "encoder_emb_dim": 64,
    "encoder_conv_channels": (128, 256, 512),
    "encoder_kernel_sizes": (9, 9, 11),
    "encoder_dropout_rate": 0.2,
    "prediction_head_hidden_dims": (512, 512, 512),
    "prediction_head_dropout_rate": 0.1}

# init the dataloaders
train_loader = create_dataloader("../step_1_data_preparation/chembl224_ki_train_aug.h5",
                                 batch_size=base_config["batch_size"],
                                 shuffle=True,
                                 to_device=True)

val_loader = create_dataloader("../step_1_data_preparation/chembl224_ki_val_aug.h5",
                               batch_size=base_config["batch_size"],
                               shuffle=True,
                               to_device=True)

In [4]:
import json

ckpt_dir = (Path.cwd() / base_config["model_save_path"]).resolve()
ckpt_dir.parent.mkdir(parents=True, exist_ok=True)
checkpointer = ocp.PyTreeCheckpointer()
global_best = {"val_loss": float("inf"), "config": None}


def objective(trial):

    optuna_suggestions = {"prediction_head_n_layers": trial.suggest_int("prediction_head_n_layers", 2, 4),
                          "prediction_head_hidden_dim": trial.suggest_categorical("prediction_head_hidden_dim", [256, 512, 768, 1024]),
                          "lr": trial.suggest_float("lr", 1e-5, 1e-2, log=True),
                          "prediction_head_dropout_rate": trial.suggest_float("prediction_head_dropout_rate", 0.0, 0.4)}

    config = base_config.copy() | optuna_suggestions

    print(f"\nTrial {trial.number} with config: {optuna_suggestions}")

    # init the model
    encoder = SmilesEncoder(vocab_size=config["vocab_size"],
                            latent_dim=config["latent_dim"],
                            emb_dim=config["encoder_emb_dim"],
                            conv_channels=config["encoder_conv_channels"],
                            kernel_sizes=config["encoder_kernel_sizes"],
                            dropout_rate=config["encoder_dropout_rate"])

    prediction_head = PredictionHead(latent_dim=config["latent_dim"],
                            hidden_dims=tuple(config["prediction_head_hidden_dim"] for _ in range(config["prediction_head_n_layers"])),
                            dropout_rate=config["prediction_head_dropout_rate"])

    model = EncoderPredictor(encoder, prediction_head)
    optimizer = nnx.Optimizer(model, optax.adamw(config["lr"]), wrt=nnx.Param)

    # Training loop with early stopping
    best_val_loss = float("inf")
    no_improve = 0

    for epoch in range(config["n_epochs"]):
        epoch_train_loss = 0.0
        epoch_val_loss = 0.0

        for train_batch in train_loader:
            train_loss = oracle_train_step(model, optimizer, train_batch)
            epoch_train_loss += float(train_loss)

        for val_batch in val_loader:
            val_loss = oracle_val_step(model, val_batch)
            epoch_val_loss += float(val_loss)

        epoch_train_loss /= len(train_loader)
        epoch_val_loss /= len(val_loader)

        if epoch % 5 == 0:
            print(f"\tEpoch {epoch}: train loss = {epoch_train_loss:.4f}, val loss = {epoch_val_loss:.4f}")

        # early stopping within trial
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            no_improve = 0
            # save checkpoint + config only if this is the best model across ALL trials
            if best_val_loss < global_best["val_loss"]:
                global_best["val_loss"] = best_val_loss
                global_best["config"] = config
                checkpointer.save(str(ckpt_dir), nnx.state(model), force=True)
                with open(ckpt_dir.parent / "best_oracle_config.json", "w") as f:
                    json.dump({k: list(v) if isinstance(v, tuple) else v for k, v in config.items()}, f, indent=2)
        else:
            no_improve += 1
            if no_improve >= config["early_stop_patience_evals"]:
                print(f"\tEarly stopping at epoch {epoch} (best val loss: {best_val_loss:.4f})")
                break

        # Optuna pruning
        trial.report(epoch_val_loss, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return best_val_loss


# --- Run ---

study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=20),
)
study.optimize(objective, n_trials=50)

print(f"\nBest val loss: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")
print(f"Checkpoint saved at: {ckpt_dir}")

[I 2026-03-24 18:16:02,107] A new study created in memory with name: no-name-97046c0b-77dd-4738-b3c7-2e80a8d75c46



Trial 0 with config: {'prediction_head_n_layers': 4, 'prediction_head_hidden_dim': 256, 'lr': 0.00020201538295547814, 'prediction_head_dropout_rate': 0.09571039519403098}
	Epoch 0: train loss = 18.5464, val loss = 49.4241
	Epoch 5: train loss = 0.9273, val loss = 16.0597
	Epoch 10: train loss = 0.7767, val loss = 2.0284
	Epoch 15: train loss = 0.6407, val loss = 1.6641
	Epoch 20: train loss = 0.5789, val loss = 1.3177


[I 2026-03-24 18:18:57,066] Trial 0 finished with value: 0.6064920286337535 and parameters: {'prediction_head_n_layers': 4, 'prediction_head_hidden_dim': 256, 'lr': 0.00020201538295547814, 'prediction_head_dropout_rate': 0.09571039519403098}. Best is trial 0 with value: 0.6064920286337535.


	Early stopping at epoch 24 (best val loss: 0.6065)

Trial 1 with config: {'prediction_head_n_layers': 3, 'prediction_head_hidden_dim': 768, 'lr': 0.0002598838505425031, 'prediction_head_dropout_rate': 0.07568166676686534}
	Epoch 0: train loss = 6.6657, val loss = 48.4938
	Epoch 5: train loss = 0.7392, val loss = 36.8361
	Epoch 10: train loss = 0.6005, val loss = 5.3992
	Epoch 15: train loss = 0.4762, val loss = 0.7684
	Epoch 20: train loss = 0.3879, val loss = 0.7009
	Epoch 25: train loss = 0.3359, val loss = 1.3058


[I 2026-03-24 18:22:22,216] Trial 1 finished with value: 0.5870690027872721 and parameters: {'prediction_head_n_layers': 3, 'prediction_head_hidden_dim': 768, 'lr': 0.0002598838505425031, 'prediction_head_dropout_rate': 0.07568166676686534}. Best is trial 1 with value: 0.5870690027872721.


...

	Early stopping at epoch 24 (best val loss: 0.5933)

Trial 48 with config: {'prediction_head_n_layers': 3, 'prediction_head_hidden_dim': 512, 'lr': 0.00019459050715959117, 'prediction_head_dropout_rate': 0.0001689477747556234}
	Epoch 0: train loss = 11.0442, val loss = 50.6523
	Epoch 5: train loss = 0.6664, val loss = 28.1555
	Epoch 10: train loss = 0.4953, val loss = 5.2281
	Epoch 15: train loss = 0.3702, val loss = 0.9419
	Epoch 20: train loss = 0.2999, val loss = 0.5994


[I 2026-03-24 20:37:34,404] Trial 48 finished with value: 0.581927266716957 and parameters: {'prediction_head_n_layers': 3, 'prediction_head_hidden_dim': 512, 'lr': 0.00019459050715959117, 'prediction_head_dropout_rate': 0.0001689477747556234}. Best is trial 45 with value: 0.5480707680185636.


	Early stopping at epoch 24 (best val loss: 0.5819)

Trial 49 with config: {'prediction_head_n_layers': 4, 'prediction_head_hidden_dim': 512, 'lr': 0.00014878760838126455, 'prediction_head_dropout_rate': 0.03351986486492415}
	Epoch 0: train loss = 14.6617, val loss = 51.8825
	Epoch 5: train loss = 0.7564, val loss = 25.2927
	Epoch 10: train loss = 0.5582, val loss = 3.9263
	Epoch 15: train loss = 0.4360, val loss = 1.6578
	Epoch 20: train loss = 0.3527, val loss = 0.8322


[I 2026-03-24 20:40:40,552] Trial 49 finished with value: 0.5639536033074061 and parameters: {'prediction_head_n_layers': 4, 'prediction_head_hidden_dim': 512, 'lr': 0.00014878760838126455, 'prediction_head_dropout_rate': 0.03351986486492415}. Best is trial 45 with value: 0.5480707680185636.


	Early stopping at epoch 24 (best val loss: 0.5640)

Best val loss: 0.5481
Best params: {'prediction_head_n_layers': 3, 'prediction_head_hidden_dim': 512, 'lr': 0.00012498934785079352, 'prediction_head_dropout_rate': 0.03627648144417609}
Checkpoint saved at: /Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/project/step_4_train_oracle/checkpoints/pretrained_oracle


In [5]:
# Load config + checkpoint
ckpt_dir = (Path.cwd() / base_config["model_save_path"]).resolve()
config_path = ckpt_dir.parent / "best_oracle_config.json"

with open(config_path) as f:
    best_config = json.load(f)

# rebuild the model with the saved architecture
encoder = SmilesEncoder(vocab_size=best_config["vocab_size"],
                        latent_dim=best_config["latent_dim"],
                        emb_dim=best_config["encoder_emb_dim"],
                        conv_channels=tuple(best_config["encoder_conv_channels"]),
                        kernel_sizes=tuple(best_config["encoder_kernel_sizes"]),
                        dropout_rate=best_config["encoder_dropout_rate"])

prediction_head = PredictionHead(latent_dim=best_config["latent_dim"],
                        hidden_dims=tuple(best_config["prediction_head_hidden_dim"] for _ in range(best_config["prediction_head_n_layers"])),
                        dropout_rate=best_config["prediction_head_dropout_rate"])

model = EncoderPredictor(encoder, prediction_head)

checkpointer = ocp.PyTreeCheckpointer()
restored_state = checkpointer.restore(str(ckpt_dir), item=nnx.state(model))
nnx.update(model, restored_state)

# sanity check
test_batch = next(iter(val_loader))
restored_val_loss = float(oracle_val_step(model, test_batch))
print(f"Restored model val loss: {restored_val_loss:.4f}")
print(f"Config: {best_config}")

/Users/derekvantilborg/Dropbox/coding/OOD-proof-mol-RL/.venv/lib/python3.13/site-packages/orbax/checkpoint/_src/serialization/jax_array_handlers.py:712: UserWarning: Sharding info not provided when restoring. Populating sharding info from sharding file. Please note restoration time will be slightly increased due to reading from file. Note also that this option is unsafe when restoring on a different topology than the checkpoint was saved with.
  warnings.warn(


Restored model val loss: 0.5731
Config: {'dataset': 'chembl224_ki', 'batch_size': 256, 'n_epochs': 2000, 'early_stop_patience_evals': 5, 'model_save_path': 'checkpoints/pretrained_oracle', 'vocab_size': 36, 'max_seq_len': 102, 'latent_dim': 128, 'encoder_emb_dim': 64, 'encoder_conv_channels': [128, 256, 512], 'encoder_kernel_sizes': [9, 9, 11], 'encoder_dropout_rate': 0.2, 'prediction_head_hidden_dims': [512, 512, 512], 'prediction_head_dropout_rate': 0.03627648144417609, 'prediction_head_n_layers': 3, 'prediction_head_hidden_dim': 512, 'lr': 0.00012498934785079352}
